In [0]:
# ============================================================
# 06_ORCHESTRATION
# Verify full pipeline: Bronze → Silver → Gold → Predictions
# ============================================================

tables = {
    "Bronze": "workspace.default.bronze_loan_applications",
    "Silver": "workspace.default.silver_loan_applications",
    "Gold"  : "workspace.default.gold_loan_features",
    "Preds" : "workspace.default.gold_loan_predictions"
}

print("=" * 55)
print("   LOAN DEFAULT PREDICTION — PIPELINE VERIFICATION")
print("=" * 55)

for layer, table in tables.items():
    try:
        count = spark.table(table).count()
        cols  = len(spark.table(table).columns)
        print(f"✅ {layer:8s} | Rows: {count:>7,} | Cols: {cols:>3}")
    except Exception as e:
        print(f"❌ {layer:8s} | ERROR: {e}")

   LOAN DEFAULT PREDICTION — PIPELINE VERIFICATION
✅ Bronze   | Rows: 307,511 | Cols: 125
✅ Silver   | Rows: 307,511 | Cols:  75
✅ Gold     | Rows: 307,511 | Cols:  89
✅ Preds    | Rows:  61,503 | Cols:  37


In [0]:
from pyspark.sql.functions import col

print("\n📊 PIPELINE LINEAGE SUMMARY")
print("-" * 55)

bronze = spark.table(tables["Bronze"])
silver = spark.table(tables["Silver"])
gold   = spark.table(tables["Gold"])
preds  = spark.table(tables["Preds"])

bronze_rows = bronze.count()
silver_rows = silver.count()
gold_rows   = gold.count()
pred_rows   = preds.count()

print(f"  Bronze  →  {bronze_rows:>7,} rows | {len(bronze.columns):>3} cols (raw ingestion)")
print(f"  Silver  →  {silver_rows:>7,} rows | {len(silver.columns):>3} cols (cleaned)")
print(f"  Gold    →  {gold_rows:>7,} rows | {len(gold.columns):>3} cols (features)")
print(f"  Preds   →  {pred_rows:>7,} rows | {len(preds.columns):>3} cols (predictions)")
print("-" * 55)
print(f"  Rows retained through pipeline: {round(silver_rows/bronze_rows*100, 1)}%")


📊 PIPELINE LINEAGE SUMMARY
-------------------------------------------------------
  Bronze  →  307,511 rows | 125 cols (raw ingestion)
  Silver  →  307,511 rows |  75 cols (cleaned)
  Gold    →  307,511 rows |  89 cols (features)
  Preds   →   61,503 rows |  37 cols (predictions)
-------------------------------------------------------
  Rows retained through pipeline: 100.0%


In [0]:
from pyspark.sql.functions import sum as spark_sum, col

print("\n🔍 DATA QUALITY CHECKS")
print("-" * 55)

# Check 1: No nulls in TARGET column
null_target = gold.filter(col("TARGET").isNull()).count()
print(f"  {'✅' if null_target == 0 else '❌'} Null TARGET values     : {null_target}")

# Check 2: No negative credit amounts
neg_credit = gold.filter(col("AMT_CREDIT") < 0).count()
print(f"  {'✅' if neg_credit == 0 else '❌'} Negative credit amounts: {neg_credit}")

# Check 3: Predictions table has both classes
pred_classes = preds.select("PREDICTED_DEFAULT").distinct().count()
print(f"  {'✅' if pred_classes == 2 else '❌'} Prediction classes     : {pred_classes} (expected 2)")

# Check 4: Probability range valid
invalid_probs = preds.filter(
    (col("DEFAULT_PROBABILITY") < 0) | (col("DEFAULT_PROBABILITY") > 1)
).count()
print(f"  {'✅' if invalid_probs == 0 else '❌'} Invalid probabilities  : {invalid_probs}")

# Check 5: Gold has engineered features
has_ratio = "CREDIT_INCOME_RATIO" in gold.columns
print(f"  {'✅' if has_ratio else '❌'} Engineered features exist")

print("-" * 55)
print("  ✅ All data quality checks passed!")


🔍 DATA QUALITY CHECKS
-------------------------------------------------------
  ✅ Null TARGET values     : 0
  ✅ Negative credit amounts: 0
  ✅ Prediction classes     : 2 (expected 2)
  ✅ Invalid probabilities  : 0
  ✅ Engineered features exist
-------------------------------------------------------
  ✅ All data quality checks passed!


In [0]:
from mlflow.tracking import MlflowClient
import mlflow

print("\n🤖 MLFLOW MODEL SUMMARY")
print("-" * 55)

client = MlflowClient()
experiment = client.get_experiment_by_name("/loan_default_prediction")
runs = client.search_runs(
    experiment.experiment_id,
    order_by=["metrics.roc_auc DESC"]
)

for i, run in enumerate(runs[:3]):
    m = run.data.metrics
    p = run.data.params
    print(f"\n  Run #{i+1}: {run.info.run_name}")
    print(f"    ROC-AUC   : {m.get('roc_auc', 'N/A')}")
    print(f"    Accuracy  : {m.get('accuracy', 'N/A')}")
    print(f"    F1 Score  : {m.get('f1_score', 'N/A')}")
    print(f"    Precision : {m.get('precision', 'N/A')}")
    print(f"    Recall    : {m.get('recall', 'N/A')}")


🤖 MLFLOW MODEL SUMMARY
-------------------------------------------------------

  Run #1: RandomForest_v1
    ROC-AUC   : 0.7345335563241036
    Accuracy  : 0.7199323610230395
    F1 Score  : 0.261078460812492
    Precision : 0.16586721901231877
    Recall    : 0.6128902316213495

  Run #2: RandomForest_v1
    ROC-AUC   : 0.7345335563241036
    Accuracy  : 0.7199323610230395
    F1 Score  : 0.261078460812492
    Precision : 0.16586721901231877
    Recall    : 0.6128902316213495


In [0]:
print("""
╔══════════════════════════════════════════════════════════╗
║         PIPELINE ORCHESTRATION — COMPLETE ✅             ║
╠══════════════════════════════════════════════════════════╣
║  NOTEBOOKS EXECUTED (in order):                          ║
║  01_bronze_ingestion      ✅ Raw CSV → Bronze Delta      ║
║  02_silver_cleaning       ✅ Clean + Fix → Silver Delta  ║
║  03_gold_features         ✅ Features → Gold Delta       ║
║  04_ml_training           ✅ Train + MLflow + Register   ║
║  05_eda_business_insights ✅ SQL Analytics + Insights    ║
║  06_orchestration         ✅ Verify + Quality Checks     ║
╠══════════════════════════════════════════════════════════╣
║  ARCHITECTURE: Medallion (Bronze → Silver → Gold)        ║
║  STORAGE     : Delta Lake (Unity Catalog)                ║
║  ML TRACKING : MLflow (Databricks managed)               ║
║  PLATFORM    : Databricks Serverless                     ║
╚══════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════╗
║         PIPELINE ORCHESTRATION — COMPLETE ✅             ║
╠══════════════════════════════════════════════════════════╣
║  NOTEBOOKS EXECUTED (in order):                          ║
║  01_bronze_ingestion      ✅ Raw CSV → Bronze Delta      ║
║  02_silver_cleaning       ✅ Clean + Fix → Silver Delta  ║
║  03_gold_features         ✅ Features → Gold Delta       ║
║  04_ml_training           ✅ Train + MLflow + Register   ║
║  05_eda_business_insights ✅ SQL Analytics + Insights    ║
║  06_orchestration         ✅ Verify + Quality Checks     ║
╠══════════════════════════════════════════════════════════╣
║  ARCHITECTURE: Medallion (Bronze → Silver → Gold)        ║
║  STORAGE     : Delta Lake (Unity Catalog)                ║
║  ML TRACKING : MLflow (Databricks managed)               ║
║  PLATFORM    : Databricks Serverless                     ║
╚══════════════════════════════════════════════════════════╝

